# Grid, random and Optuna search

This offline notebook demonstrates discrete candidates, continuous ranges, repeated-run objectives, and copying a selected configuration into an experiment. Small budgets teach the API; increase them for a research study.

In [ ]:
import numpy as np
import pandas as pd
from polyclustx.data import MultiViewDataset, simulate_missing_views
from polyclustx.experiment import Experiment, GridSearchExperiment, RandomSearchExperiment, OptunaSearchExperiment
from polyclustx.metrics import clustering_accuracy, nmi_score, silhouette_score
from polyclustx.models import (LMGEC, SCM, IDEC, COMPLETER, DCP, MFLVC, CONAN, DGF,
                         SGLA, UMC, GCoMVKM, MultiViewKMeans, MultiViewSpectralClustering)

def dataset(name, noise, seed):
    rng = np.random.default_rng(seed)
    labels = np.repeat(np.arange(3), 16)
    views = [rng.normal(size=(3, dim))[labels] * 3 +
             rng.normal(scale=noise, size=(48, dim)) for dim in (6, 8)]
    return MultiViewDataset(views, labels=labels, name=name)

clean = dataset("clean", .3, 1)
noisy = dataset("noisy", 1.5, 2)

In [ ]:
space = {"n_clusters": [3], "embedding_dim": [3, 4],
         "temperature": [.2, 1., 3.], "max_iter": [3]}
common = dict(model_class=LMGEC, dataset=clean, metrics={"NMI": nmi_score},
              best_metric="NMI", runs=2, seed=42, save_path="experiments/searches")
grid = GridSearchExperiment(param_grid=space, **common).run()
random = RandomSearchExperiment(param_distributions=space, n_iter=3, **common).run()

## Adaptive sampling

Lists mean exact choices. A dictionary with low/high gives a continuous range; `type='int'` gives integers. Optional logarithmic sampling requires positive bounds. Optuna learns from completed mean scores. It does not prune fits.

In [ ]:
adaptive = OptunaSearchExperiment(
    param_distributions={
        "n_clusters": [3],
        "embedding_dim": {"type": "int", "low": 2, "high": 4},
        "temperature": {"low": .1, "high": 5., "log": True},
        "max_iter": [3],
    },
    n_iter=4, **common,
)
optuna_result = adaptive.run()
display(pd.DataFrame(optuna_result.report["search"]["steps"]))
optuna_result.export("experiments/searches/optuna.zip")

## Run the selected configuration

Search best parameters are regular constructor values. The search objective is measured on these data, not on an independent held-out test set. Avoid presenting it as unbiased generalization performance.

In [ ]:
if optuna_result.best_params:
    best_model = LMGEC(**optuna_result.best_params)
    repeated = Experiment(clean, [best_model], {"ACC": clustering_accuracy, "NMI": nmi_score},
                          runs=3, seed=100, save_path="experiments/best-model").run()

## Search without true labels

Silhouette uses the learned embedding and predicted clusters. Ground-truth labels are ignored when benchmark=False. Davies–Bouldin can also be selected and is minimized automatically.

In [ ]:
internal = OptunaSearchExperiment(
    MultiViewKMeans, {"n_clusters": [2, 3, 4], "n_init": [1, 3], "max_iter": [10]},
    clean, metrics={"Silhouette": silhouette_score}, best_metric="Silhouette",
    benchmark=False, n_iter=3, runs=2, seed=42, save_path="experiments/internal-search",
).run()
print(internal.best_params)